```sh
scp oci-nrt-cs-001-dc-02:dev/nemo-rl-n5p5-mmpr-tiny/mmpr_grading.jsonl .
```

In [1]:
import json
import random
from PIL import Image
import re
import pandas as pd
import plotly.express as px
import numpy as np
from glob import glob
from mathruler.grader import grade_answer
from tqdm import tqdm


In [ ]:
with open("mmpr_grading.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

df = pd.DataFrame(data)

In [3]:
df.head()

,id,dataset,truncated,answered,score,verifier
0,13000004081,mmpr-1.2-geomverse_extracted_pairs_vqa_correct...,False,True,1.0,mathruler
1,10800001899,mmpr-1.2-geometry3k_en_20240402_extracted_open...,False,True,1.0,mathruler
2,13200025394,mmpr-1.2-geo170k_extracted_pairs_vqa_correctne...,False,False,0.0,unanswered
3,11600001265,mmpr-1.2-geos_en_20240402_extracted_pairs_vqa_...,False,True,1.0,mathruler
4,10000010285,mmpr-1.2-ai2d_train_12k_en_20240410_extracted_...,False,True,1.0,string-match


In [4]:
x=df.groupby(["dataset", "verifier", "id"]).agg({"score": "mean"}).reset_index()

In [5]:
counts = df.groupby(["id"]).agg({"score": "mean", "truncated": "mean"}).reset_index()
counts["score"] = counts["score"].apply(lambda x: 0 if x == 0 else max(1, round(5 * x)) / 5)
counts.loc[counts["truncated"] == 1, "score"] = -1


In [44]:
counts = df.groupby(["id"]).agg({"score": "mean", "truncated": "mean"}).reset_index()
counts["score"] = counts["score"].apply(lambda x: 0 if x == 0 else max(1, round(5 * x)) / 5)
# counts.loc[counts["truncated"] == 1, "score"] = -0.1
counts = counts.groupby(["score"]).size().reset_index(name="value")
counts["pct"] = counts["value"] / counts["value"].sum()
fig = px.bar(counts, x="score", y="pct", range_x=[-0.1, 1.1], width=800, title="MMPR-1.2, 5 samples per prompt (N=493k)")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="percentage")
fig

In [34]:
# save dataset of samples with 20–80% correct
# 1. sort each dataset by increasing difficulty
# 2. break ties randomly
# 3. merge datasets randomly

train_ids = df.groupby(["id"]).agg({"score": "mean", "dataset": "first"}).reset_index()
train_ids = train_ids[(train_ids["score"] > 0) & (train_ids["score"] < 1)]
train_ids["random"] = np.random.rand(len(train_ids))
train_ids.sort_values(by=["score", "random"], ascending=False, inplace=True)

train_ids2 = train_ids.copy()
for dataset in train_ids["dataset"].unique():
    train_ids2.loc[train_ids2["dataset"] == dataset] = train_ids[train_ids["dataset"] == dataset]

train_ids2[["id"]].to_csv("mmpr_train_ids.csv", index=False, header=False)
len(train_ids2)

160693

In [35]:
train_ids2

,id,score,dataset,random
45802,500045803,0.80,mmpr-1.2-nlvr2_en_20240910_ov_pairs_vqa_format...,0.999950
311990,3600311991,0.80,mmpr-1.2-tabmwp_en_20240402_cot_pairs_vqa_corr...,0.999898
460681,13200462252,0.80,mmpr-1.2-geo170k_extracted_pairs_vqa_correctne...,0.999896
162336,3400162337,0.80,mmpr-1.2-vsr_en_20240402_cot_ques_pairs_vqa_co...,0.999855
404038,10000405609,0.80,mmpr-1.2-ai2d_train_12k_en_20240410_extracted_...,0.999846
...,...,...,...,...
99278,600099279,0.02,mmpr-1.2-inat_train2018_merge_en_20240811_sr0....,0.522726
104543,600104544,0.02,mmpr-1.2-inat_train2018_merge_en_20240811_sr0....,0.519135
87193,600087194,0.02,mmpr-1.2-inat_train2018_merge_en_20240811_sr0....,0.471041
91556,600091557,0.02,mmpr-1.2-inat_train2018_merge_en_20240811_sr0....,0.379552


In [19]:
from mathruler.grader import grade_answer

grade_answer("800000", "800,000")

True

In [26]:
counts = df.groupby(["dataset", "id"]).agg({"score": "mean"}).reset_index()
counts["score"] = counts["score"].apply(lambda x: round(5 * x) / 5)
counts = counts.groupby(["dataset", "score"]).size().reset_index(name="value")
counts["pct"] = counts.groupby("dataset")["value"].transform(lambda v: v / v.sum())
counts.sort_values(by="pct", ascending=True, inplace=True)
fig = px.bar(counts[counts["score"]==0], y="dataset", x="pct", orientation="h", width=1000, height=2000)
            #  range_x=[-0.1, 1.1], width=1400, title="SFT math, 5 samples per prompt (N=106k)")
fig.update_layout(xaxis_tickformat=".0%", xaxis_title="percentage")
fig


In [17]:
df_plot

,dataset,score,value,pct
0,mmpr-1.2-CLEVR_math_en_20240402_extracted_pair...,0.0,20,0.018282
1,mmpr-1.2-CLEVR_math_en_20240402_extracted_pair...,0.2,21,0.019196
2,mmpr-1.2-CLEVR_math_en_20240402_extracted_pair...,0.4,57,0.052102
3,mmpr-1.2-CLEVR_math_en_20240402_extracted_pair...,0.6,93,0.085009
4,mmpr-1.2-CLEVR_math_en_20240402_extracted_pair...,0.8,178,0.162706
...,...,...,...,...
712,mmpr-1.2-vsr_en_20240402_cot_ques_pairs_vqa_co...,0.2,7513,0.071011
713,mmpr-1.2-vsr_en_20240402_cot_ques_pairs_vqa_co...,0.4,6161,0.058232
714,mmpr-1.2-vsr_en_20240402_cot_ques_pairs_vqa_co...,0.6,6784,0.064120
715,mmpr-1.2-vsr_en_20240402_cot_ques_pairs_vqa_co...,0.8,9816,0.092778


In [15]:
df_plot = (
    df.groupby(["dataset", "id"]).agg({"score": "mean"}).reset_index()
              .groupby(["dataset", "score"]).size().reset_index(name="value")
)
df_plot["pct"] = df_plot.groupby("dataset")["value"].transform(lambda v: v / v.sum())
fig = px.bar(df_plot, x="score", y="pct", color="dataset", barmode="group",
             range_x=[-0.1, 1.1], width=1400, title="SFT math, 5 samples per prompt (N=106k)")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="percentage")
fig


In [11]:
counts_correct = (
    df_correct.groupby(["id"]).agg({"score": "mean"}).reset_index()
       .groupby(["score"]).size().reset_index(name="value")
)
counts_correct["pct"] = counts_correct["value"] / counts_correct["value"].sum()
fig = px.bar(counts_correct, x="score", y="pct", range_x=[-0.1, 1.1], width=800, title="SFT math one-shot correct samples (N=86k)")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="percentage")
fig

In [12]:
df_plot = (
    df_correct.groupby(["dataset", "id"]).agg({"score": "mean"}).reset_index()
              .groupby(["dataset", "score"]).size().reset_index(name="value")
)
df_plot["pct"] = df_plot.groupby("dataset")["value"].transform(lambda v: v / v.sum())
fig = px.bar(df_plot, x="score", y="pct", color="dataset", barmode="group",
             range_x=[-0.1, 1.1], width=800, title="SFT math one-shot correct samples (N=86k)")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="percentage")
fig


In [13]:
counts_incorrect = (
    df_incorrect.groupby(["id"]).agg({"score": "mean"}).reset_index()
       .groupby(["score"]).size().reset_index(name="value")
)
counts_incorrect["pct"] = counts_incorrect["value"] / counts_incorrect["value"].sum()
fig = px.bar(counts_incorrect, x="score", y="pct", range_x=[-0.1, 1.1], width=800, title="SFT math one-shot incorrect samples (N=20k)")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="percentage")
fig


In [14]:
df_plot = (
    df_incorrect.groupby(["dataset", "id"]).agg({"score": "mean"}).reset_index()
                .groupby(["dataset", "score"]).size().reset_index(name="value")
)
df_plot["pct"] = df_plot.groupby("dataset")["value"].transform(lambda v: v / v.sum())
fig = px.bar(df_plot, x="score", y="pct", color="dataset", barmode="group",
             range_x=[-0.1, 1.1], width=800, title="SFT math one-shot incorrect samples (N=20k)")
fig.update_layout(yaxis_tickformat=".0%", yaxis_title="percentage")
fig


In [135]:
df=pd.merge(df_full, df_full.groupby("id").agg({"score": "mean"}).reset_index(), on="id", how="inner", suffixes=("", "_mean"))
df[(df["dataset"] == "geomverse") & (df["score_mean"] == 0.0)][["id", "dataset", "pred_answer", "gt_answer", "image", "question", "response"]].sample(30).iloc[0].tolist()


[2000030450,
 'geomverse',
 '66.16',
 '64.8',
 '0004261.jpg',
 'If the area of the ABC sector is 100.48, the BCDEF shape is a rectangle where an equilateral triangle has been removed from one side of it, the length of the CD side is 12 and the area of the BCDEF shape is 96, compute the degree of the CBA angle. Assume $\\pi=3.14$. Round computations to 2 decimal places.\nExplain step-by-step.\nThink step-by-step and write the final answer in this format:\n\nThe answer is \\(...\\).',
 'The answer is \\(66.16\\).']

In [103]:
df[(df["dataset"] == "educhat_math") & (df["score_mean"] == 0.0)][["id", "dataset", "pred_answer", "gt_answer"]].sample(30)


,id,dataset,pred_answer,gt_answer
453119,4000030100,educhat_math,None,C
465588,4000076120,educhat_math,\times,错误
529088,4000057261,educhat_math,"\sin B = \frac{\sqrt{21}}{7}, c = 3",\frac{\sqrt{21}}{7} ; 3
529937,4000128351,educhat_math,None,答案: 60 天
498068,4000093600,educhat_math,None,13
465489,4000127640,educhat_math,"81, 2500",答案: $81 \quad 9 \quad 9 \quad 2500 \quad 50 \q...
527138,4000063951,educhat_math,None,-5
537251,4000088551,educhat_math,"100, 26",100 \quad 26
515868,4000131070,educhat_math,(1) x = -3 或 x = -9; (2) x = \frac{1}{2} 或 x =...,"x_1 = -3, x_2 = -9; x_1 = 3, x_2 = \frac{1}{2}..."
475093,4000080200,educhat_math,None,\( BC = EF \) (答案不唯一)
